In [ ]:
import getpass

client_id = input("Enter your Strava Client ID: ").strip()
client_secret = getpass.getpass("Enter your Strava Client Secret: ")

In [ ]:
from urllib.parse import urlencode
from IPython.display import display, HTML

params = {
    "client_id": client_id,
    "response_type": "code",
    "redirect_uri": "http://localhost/exchange_token",
    "approval_prompt": "force",
    "scope": "activity:read_all,activity:write"
}

auth_url = "https://www.strava.com/oauth/authorize?" + urlencode(params)

display(HTML(f'<a href="{auth_url}" target="_blank">Authorize with Strava</a>'))

In [ ]:
from urllib.parse import urlencode
from IPython.display import display, HTML

params = {
    "client_id": client_id,
    "response_type": "code",
    "redirect_uri": "http://localhost/exchange_token",
    "approval_prompt": "force",
    "scope": "read,activity:read_all,activity:write,profile:read_all"
}

auth_url = "https://www.strava.com/oauth/authorize?" + urlencode(params)

display(HTML(f'<a href="{auth_url}" target="_blank">Authorize Strava with profile access</a>'))

In [ ]:
from urllib.parse import urlparse, parse_qs
import getpass
import requests

authorization_code = getpass.getpass(
    "Paste the NEW authorization code here: "
).strip()

response = requests.post(
    "https://www.strava.com/oauth/token",
    data={
        "client_id": client_id,
        "client_secret": client_secret,
        "code": authorization_code,
        "grant_type": "authorization_code"
    },
    timeout=30
)

if response.ok:
    tokens = response.json()
    print("Strava authentication successful.")
    print("Token received and ready for the next step.")
else:
    print("Token exchange failed:", response.status_code)
    print(response.text)

In [ ]:
import requests

access_token = tokens["access_token"]
headers = {"Authorization": f"Bearer {access_token}"}

response = requests.get(
    "https://www.strava.com/api/v3/athlete",
    headers=headers,
    timeout=30
)

if response.ok:
    athlete = response.json()

    print("Strava profile loaded.")
    print("\nShoes:")

    for shoe in athlete.get("shoes", []):
        print(f"Name: {shoe['name']}| Gear ID: {shoe['id']}")

else:
    print("Request failed:", response.status_code)
    print(response.text)

In [ ]:
import requests
import pandas as pd
import time

headers = {"Authorization": f"Bearer {tokens['access_token']}"}

all_activities = []
page = 1

while True:
    response = requests.get(
        "https://www.strava.com/api/v3/athlete/activities",
        headers=headers,
        params={"per_page": 200, "page": page},
        timeout=30
    )

    if not response.ok:
        print("Request failed:", response.status_code)
        print(response.text)
        break

    batch = response.json()

    if not batch:
        break

    all_activities.extend(batch)
    print(f"Page {page}: retrieved {len(batch)} activities")

    page += 1
    time.sleep(0.2)

walks = [
    a for a in all_activities
    if a.get("type") == "Walk" or a.get("sport_type") == "Walk"
]

print("\nTotal activities retrieved:", len(all_activities))
print("Total Walk activities:", len(walks))

walk_df = pd.DataFrame([
    {
        "Date": a.get("start_date_local"),
        "Activity": a.get("name"),
        "Activity ID": a.get("id"),
        "Current Gear ID": a.get("gear_id")
    }
    for a in walks
])

display(walk_df)

In [ ]:
import pandas as pd
import glob
import os

csv_files = glob.glob("/content/*.csv")

if not csv_files:
    print("No CSV file found. Please upload the file to Colab.")
else:
    print("CSV files found:")
    for i, file in enumerate(csv_files):
        print(i, os.path.basename(file))

In [ ]:
import pandas as pd

csv_path = "/content/InteractiveSheet_2026-10-29_15_25_04 - Sheet1.csv"

df = pd.read_csv(csv_path)

print("Columns:", df.columns.tolist())
print("Total rows:", len(df))
print("\nFirst 5 rows:")
display(df.head())

print("\nMissing values by column:")
print(df.isna().sum())

In [ ]:
valid_gear_ids = {
    "g33895129",
    "g33895117",
    "g33895090",
    "g32972043",
    "g34640092"
}

df["New Shoe"] = df["New Shoe"].fillna("").astype(str).str.strip()
df["Current Gear ID"] = (
    df["Current Gear ID"].fillna("").astype(str).str.strip()
)

assigned = df[df["New Shoe"] != ""].copy()

invalid = assigned[~assigned["New Shoe"].isin(valid_gear_ids)]
already_correct = assigned[
    assigned["New Shoe"] == assigned["Current Gear ID"]
]
needs_change = assigned[
    assigned["New Shoe"] != assigned["Current Gear ID"]
]

print("Total activities:", len(df))
print("Selected assignments:", len(assigned))
print("Invalid gear IDs:", len(invalid))
print("Already using selected gear:", len(already_correct))
print("Potential changes:", len(needs_change))

if not invalid.empty:
    display(invalid[["Date", "Activity", "Activity ID", "New Shoe"]])

In [ ]:
preview = needs_change[
    ["Date", "Activity", "Activity ID", "Current Gear ID", "New Shoe"]
].copy()

gear_names = {
    "g33895129": "Adidas Duramo Speed",
    "g33895117": "Skechers Max Run",
    "g33895090": "Adidas Ultraboost 5x",
    "g32972043": "La Sportiva Ultra Raptor 2 Mid GTX",
    "g34640092": "Reebok Terra Lite"
}

preview["Current Shoe"] = (
    preview["Current Gear ID"].map(gear_names).fillna("No gear assigned")
)
preview["Proposed Shoe"] = preview["New Shoe"].map(gear_names)

preview = preview[
    ["Date", "Activity", "Activity ID", "Current Shoe", "Proposed Shoe"]
]

print("Proposed changes:", len(preview))
display(preview.head(20))

preview.to_csv("/content/strava_gear_change_preview.csv", index=False)
print("\nFull preview saved to /content/strava_gear_change_preview.csv")

In [ ]:
import pandas as pd
import requests
import time
import os

csv_path = "/content/InteractiveSheet_2026-10-29_15_25_04 - Sheet1.csv"
log_path = "/content/strava_gear_update_log.csv"

gear_ids = {
    "g33895129",
    "g33895117",
    "g33895090",
    "g32972043",
    "g34640092"
}

df_update = pd.read_csv(csv_path, dtype={"Activity ID": str})
df_update["New Shoe"] = df_update["New Shoe"].fillna("").astype(str).str.strip()
df_update["Current Gear ID"] = df_update["Current Gear ID"].fillna("").astype(str).str.strip()

changes = df_update[df_update["New Shoe"] != ""].copy()

if not set(changes["New Shoe"]).issubset(gear_ids):
    raise ValueError("Stopped: an unrecognized gear ID was found.")

if changes["Activity ID"].duplicated().any():
    raise ValueError("Stopped: duplicate activity IDs found in the CSV.")

if len(changes) != 119:
    raise ValueError(f"Stopped: expected 119 selected rows, found {len(changes)}.")

confirm = input("Type APPLY to update the 119 approved activities: ").strip()

if confirm != "APPLY":
    print("Cancelled. No Strava updates were sent.")
else:
    headers = {"Authorization": f"Bearer {tokens['access_token']}"}
    results = []

    for _, row in changes.iterrows():
        activity_id = row["Activity ID"]
        gear_id = row["New Shoe"]

        try:
            r = requests.put(
                f"https://www.strava.com/api/v3/activities/{activity_id}",
                headers=headers,
                data={"gear_id": gear_id},
                timeout=30
            )

            results.append({
                "Activity ID": activity_id,
                "New Gear ID": gear_id,
                "HTTP Status": r.status_code,
                "Result": "Success" if r.ok else "Failed",
                "Details": "" if r.ok else r.text[:500]
            })

            pd.DataFrame(results).to_csv(log_path, index=False)

            if r.status_code == 429:
                print("Rate limit reached. Stopping to avoid further requests.")
                break

            if not r.ok:
                print(f"Failed activity {activity_id}: HTTP {r.status_code}")

            time.sleep(1)

        except requests.RequestException as e:
            results.append({
                "Activity ID": activity_id,
                "New Gear ID": gear_id,
                "HTTP Status": "",
                "Result": "Request error",
                "Details": str(e)
            })
            pd.DataFrame(results).to_csv(log_path, index=False)
            print(f"Request error for activity {activity_id}; stopping.")
            break

    results_df = pd.DataFrame(results)
    display(results_df["Result"].value_counts())
    print("Results log saved to:", log_path)

In [ ]:
import pandas as pd

csv_path = "/content/InteractiveSheet_2026-10-29_15_25_04 - Sheet1.csv"
log_path = "/content/strava_gear_update_log.csv"

df = pd.read_csv(csv_path, dtype={"Activity ID": str})
log_df = pd.read_csv(log_path, dtype={"Activity ID": str})

df["New Shoe"] = df["New Shoe"].fillna("").astype(str).str.strip()
selected = df[df["New Shoe"] != ""].copy()

successful_ids = set(
    log_df.loc[log_df["Result"] == "Success", "Activity ID"].astype(str)
)

remaining = selected[~selected["Activity ID"].isin(successful_ids)]

print("Approved assignments:", len(selected))
print("Confirmed successful:", len(successful_ids))
print("Remaining to verify/update:", len(remaining))

display(remaining[["Date", "Activity", "Activity ID", "New Shoe"]])